# Notebook 14. Elevation-Masked `850 hPa` Temperature-Gradient Sensitivity

This notebook follows the meeting guidance more directly by testing fixed high-terrain elevation masks on the `850 hPa` temperature-gradient clustering feature.

Why this notebook exists:

- the unmasked `850 hPa` temperature-gradient composite in `Notebook 10` still looked strongly terrain-linked even after display-only colorbar tightening
- simply **dropping** the `T850` feature in `Notebook 11` changed too much of the `k = 3` clustering story
- the `surface_pressure >= 900 hPa` screen in `Notebook 12` was credible but too permissive, so it changed essentially nothing in the feature or the clustering
- the meeting guidance explicitly recommended testing a true elevation mask, starting near `1000 m` and checking nearby thresholds

What changes here:

- only the `front_box_max_temp_gradient_850_tminus12_to_tplus12` feature is rebuilt
- a static surface-elevation field is derived from NOAA ETOPO1 ice-surface topography after the ARCO surface-geopotential path proved empty on this domain
- grid cells above each tested elevation threshold are set to `NaN` before the front-box maximum is computed
- the same event loop computes all three thresholds: `800 m`, `1000 m`, and `1200 m`
- the other three clustering features stay unchanged

What does **not** change here:

- `925 hPa` signed-divergence feature definitions stay unchanged
- `850 hPa` `z` anomaly logic stays unchanged
- `925 hPa` Sea-of-Japan vorticity stays unchanged
- no `300 hPa` climatology or composite rebuild is done in this notebook

Checkpointing note:

- the static surface-elevation field is cached to Drive once it is built
- the event-level elevation-masked `T850` feature table is checkpointed to Drive while it is being built so that a dropped connection does not lose all progress
- the notebook includes retained-fraction and one-event sanity checks before the long event loop so a bad mask should fail fast


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from itertools import permutations
from pathlib import Path
import gzip
import shutil
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import HOKKAIDO_FRONT_BOX, OBJECTIVE_SUBTYPE_DOMAIN
from jpcz_catalog.diagnostics import compute_temperature_gradient_magnitude, load_offset_snapshot
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.subtypes import (
    assign_hierarchical_clusters,
    compute_mean_silhouette_score,
    evaluate_hierarchical_cluster_solutions,
    standardize_feature_table,
)

RUN_EXPORT_DIR_08 = Path("outputs/verification/objective_subtype_runs")
RUN_EXPORT_DIR_08.mkdir(parents=True, exist_ok=True)
SENSITIVITY_EXPORT_DIR = Path("outputs/verification/objective_subtype_t850_elevation_mask_sensitivity")
SENSITIVITY_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_t850_elevation_mask_sensitivity_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CLUSTERED_K3_PATH = RUN_EXPORT_DIR_08 / "clustered_events_k3.csv"
PRIMARY_CLUSTER_COLUMN = "cluster_ward_3"
PRIMARY_CLUSTER_K = 3
DROPPED_FEATURE = "front_box_max_temp_gradient_850_tminus12_to_tplus12"
ELEVATION_THRESHOLDS_M = (800.0, 1000.0, 1200.0)
ETOPO1_URL = "https://www.ngdc.noaa.gov/mgg/global/relief/ETOPO1/data/ice_surface/cell_registered/netcdf/ETOPO1_Ice_c_gmt4.grd.gz"
ETOPO1_GZ_PATH = SENSITIVITY_EXPORT_DIR / "ETOPO1_Ice_c_gmt4.grd.gz"
ETOPO1_GRD_PATH = SENSITIVITY_EXPORT_DIR / "ETOPO1_Ice_c_gmt4.grd"

FULL_FEATURE_COLUMNS = [
    "coastal_to_jpcz_mean_divergence_ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    DROPPED_FEATURE,
    "sea_of_japan_mean_vorticity_peak_925",
]
MASKED_FEATURE_COLUMNS_BY_THRESHOLD = {
    int(threshold_m): [
        "coastal_to_jpcz_mean_divergence_ratio",
        "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
        f"front_box_max_temp_gradient_850_tminus12_to_tplus12_masked_{int(threshold_m)}m",
        "sea_of_japan_mean_vorticity_peak_925",
    ]
    for threshold_m in ELEVATION_THRESHOLDS_M
}
MASKED_FEATURE_COLUMN_BY_THRESHOLD = {
    int(threshold_m): f"front_box_max_temp_gradient_850_tminus12_to_tplus12_masked_{int(threshold_m)}m"
    for threshold_m in ELEVATION_THRESHOLDS_M
}
MASKED_CLUSTER_COLUMN_RAW_BY_THRESHOLD = {
    int(threshold_m): f"cluster_ward_3_masked_t850_{int(threshold_m)}m_raw"
    for threshold_m in ELEVATION_THRESHOLDS_M
}
MASKED_CLUSTER_COLUMN_BY_THRESHOLD = {
    int(threshold_m): f"cluster_ward_3_masked_t850_{int(threshold_m)}m"
    for threshold_m in ELEVATION_THRESHOLDS_M
}
CLUSTER_COUNT_OPTIONS = [2, 3, 4]
SAVE_PLOTS = True
ERA5_TIME_CHUNK = 48
FORCE_REBUILD_MASKED_T850 = False
FORCE_REBUILD_SURFACE_ELEVATION = False
CHECKPOINT_EVERY_EVENTS = 5
OFFSET_HOURS = (-12, 0, 12)

SURFACE_ELEVATION_CACHE_PATH = SENSITIVITY_EXPORT_DIR / "objective_subtype_surface_elevation.nc"
MASKED_EVENT_FEATURES_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_event_features.csv"
MASKED_EVENT_FEATURES_PARTIAL_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_event_features_partial.csv"
FEATURE_COMPARISON_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_feature_comparison_by_current_cluster.csv"
SWITCHING_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_switching_summary.csv"
CROSSTAB_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_cluster_crosstabs.csv"
COUNTS_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_cluster_counts.csv"
MEDIANS_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_cluster_medians.csv"
SOLUTION_SUMMARY_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_solution_summary.csv"
QUALITY_SCAN_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_quality_scan.csv"
VARIANCE_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_pca_variance.csv"
LOADINGS_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_pca_loadings.csv"
DRIVERS_PATH = SENSITIVITY_EXPORT_DIR / "k3_elevation_masked_t850_pca_drivers.csv"
CLUSTERED_EVENTS_PATH = SENSITIVITY_EXPORT_DIR / "clustered_events_k3_elevation_masked_t850.csv"
SURFACE_ELEVATION_DIAGNOSTICS_PATH = SENSITIVITY_EXPORT_DIR / "objective_subtype_surface_elevation_diagnostics.csv"
THRESHOLD_SUMMARY_PATH = SENSITIVITY_EXPORT_DIR / "objective_subtype_surface_elevation_threshold_summary.csv"

PCA_SCATTER_PATH = PLOT_DIR / "k3_pca_scatter_full_vs_masked_t850_thresholds.png"
SENSITIVITY_SUMMARY_PLOT_PATH = PLOT_DIR / "k3_elevation_masked_t850_sensitivity_summary.png"
T850_COMPARE_PLOT_PATH = PLOT_DIR / "k3_t850_unmasked_vs_elevation_masked_cluster_boxplots.png"

FEATURE_LABELS = {
    "coastal_to_jpcz_mean_divergence_ratio": "Coastal/JPCZ signed-divergence ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12": "Hokkaido minimum z850 anomaly",
    DROPPED_FEATURE: "Front-box maximum |grad T850|",
    "sea_of_japan_mean_vorticity_peak_925": "Sea of Japan mean 925 hPa vorticity",
}
for threshold_m, column_name in MASKED_FEATURE_COLUMN_BY_THRESHOLD.items():
    FEATURE_LABELS[column_name] = f"Front-box maximum |grad T850| (elevation mask <= {threshold_m} m)"

FEATURE_UNITS = {
    "coastal_to_jpcz_mean_divergence_ratio": "unitless",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12": "gpm",
    DROPPED_FEATURE: "K (100 km)^-1",
    "sea_of_japan_mean_vorticity_peak_925": "1e-5 s^-1",
}
for column_name in MASKED_FEATURE_COLUMN_BY_THRESHOLD.values():
    FEATURE_UNITS[column_name] = "K (100 km)^-1"


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


def axis_label(column_name: str) -> str:
    label = FEATURE_LABELS.get(column_name, column_name)
    units = FEATURE_UNITS.get(column_name, "")
    if units and units != "unitless":
        return f"{label}\n[{units}]"
    return label


def cluster_count_table(labels: pd.Series) -> pd.DataFrame:
    counts = labels.value_counts().sort_index()
    max_count = int(counts.max())
    rows = []
    for cluster_id, n_events in counts.items():
        if n_events == max_count:
            size_rank = 1
            size_descriptor = "largest subgroup"
        elif n_events == int(counts.min()):
            size_rank = int((counts.rank(method="dense", ascending=False)).loc[cluster_id])
            size_descriptor = "smallest subgroup"
        else:
            size_rank = int((counts.rank(method="dense", ascending=False)).loc[cluster_id])
            size_descriptor = "second-largest subgroup" if size_rank == 2 else f"rank {size_rank} subgroup"
        rows.append(
            {
                "cluster_id": int(cluster_id),
                "n_events": int(n_events),
                "size_rank": size_rank,
                "size_descriptor": size_descriptor,
                "cluster_label": f"Cluster {int(cluster_id)}: n={int(n_events)} ({size_descriptor})",
            }
        )
    return pd.DataFrame(rows)


def best_cluster_label_mapping(reference_labels: pd.Series, candidate_labels: pd.Series):
    reference = reference_labels.astype(int)
    candidate = candidate_labels.astype(int).reindex(reference.index)
    unique_reference = sorted(reference.dropna().unique().tolist())
    unique_candidate = sorted(candidate.dropna().unique().tolist())
    if len(unique_reference) != len(unique_candidate):
        raise ValueError(
            "Expected the same number of clusters in the reference and candidate solutions, "
            f"got {unique_reference} versus {unique_candidate}."
        )

    best_mapping = None
    best_matches = -1
    for perm in permutations(unique_reference, len(unique_candidate)):
        mapping = dict(zip(unique_candidate, perm))
        mapped = candidate.map(mapping)
        match_count = int((mapped == reference).sum())
        if match_count > best_matches:
            best_matches = match_count
            best_mapping = mapping
    return best_mapping, best_matches


def compute_pca_diagnostics(feature_df: pd.DataFrame, feature_columns: list[str]):
    standardized_df, feature_means, feature_stds = standardize_feature_table(feature_df.copy(), columns=feature_columns)
    valid = standardized_df.dropna(axis=0, how="any")
    if valid.empty:
        raise ValueError("No complete rows are available for PCA.")

    matrix = valid.to_numpy(dtype=float)
    _, singular_values, vt = np.linalg.svd(matrix, full_matrices=False)
    n_components = min(3, matrix.shape[1])
    components = vt[:n_components]
    scores = matrix @ components.T

    total_variance = float((singular_values ** 2).sum())
    explained_variance_ratio = (singular_values[:n_components] ** 2) / total_variance

    score_df = pd.DataFrame(
        scores,
        index=valid.index,
        columns=[f"PC{i + 1}" for i in range(n_components)],
    )
    variance_df = pd.DataFrame(
        {
            "principal_component": [f"PC{i + 1}" for i in range(n_components)],
            "explained_variance_ratio": explained_variance_ratio,
            "explained_variance_percent": explained_variance_ratio * 100.0,
            "cumulative_explained_variance_ratio": np.cumsum(explained_variance_ratio),
            "cumulative_explained_variance_percent": np.cumsum(explained_variance_ratio) * 100.0,
        }
    )
    loadings_df = pd.DataFrame(
        components.T,
        index=feature_columns,
        columns=[f"PC{i + 1}" for i in range(n_components)],
    ).reset_index().rename(columns={"index": "feature"})
    loadings_long_df = loadings_df.melt(id_vars="feature", var_name="principal_component", value_name="loading")
    loadings_long_df["feature_label"] = loadings_long_df["feature"].map(FEATURE_LABELS)

    driver_rows = []
    for pc_name in variance_df["principal_component"]:
        component_subset = loadings_long_df.loc[loadings_long_df["principal_component"] == pc_name].copy()
        component_subset["abs_loading"] = component_subset["loading"].abs()
        top_two = component_subset.sort_values("abs_loading", ascending=False).head(2).reset_index(drop=True)
        driver_rows.append(
            {
                "principal_component": pc_name,
                "explained_variance_percent": float(
                    variance_df.loc[variance_df["principal_component"] == pc_name, "explained_variance_percent"].iloc[0]
                ),
                "top_driver_feature": top_two.loc[0, "feature"],
                "top_driver_label": top_two.loc[0, "feature_label"],
                "top_driver_loading": float(top_two.loc[0, "loading"]),
                "second_driver_feature": top_two.loc[1, "feature"] if len(top_two) > 1 else "",
                "second_driver_label": top_two.loc[1, "feature_label"] if len(top_two) > 1 else "",
                "second_driver_loading": float(top_two.loc[1, "loading"]) if len(top_two) > 1 else np.nan,
            }
        )
    driver_df = pd.DataFrame(driver_rows)
    return standardized_df, score_df, variance_df, loadings_long_df, driver_df, feature_means, feature_stds


def subset_to_bounds(field: xr.DataArray, lon_min: float, lon_max: float, lat_min: float, lat_max: float) -> xr.DataArray:
    return field.where(
        (field.longitude >= lon_min)
        & (field.longitude <= lon_max)
        & (field.latitude >= lat_min)
        & (field.latitude <= lat_max),
        drop=True,
    )


def subset_to_box(field: xr.DataArray, box) -> xr.DataArray:
    return subset_to_bounds(field, box.lon_min, box.lon_max, box.lat_min, box.lat_max)


def _box_max(field: xr.DataArray, box) -> float:
    boxed = subset_to_box(field, box)
    return float(boxed.max(skipna=True).values)


def safe_nanmax(values) -> float:
    array = np.asarray(values, dtype=float)
    if array.size == 0 or np.all(np.isnan(array)):
        return float("nan")
    return float(np.nanmax(array))


def strip_nonspatial_coords(field: xr.DataArray) -> xr.DataArray:
    drop_coords = [coord_name for coord_name in field.coords if coord_name not in {"latitude", "longitude"}]
    if drop_coords:
        field = field.reset_coords(names=drop_coords, drop=True)
    return field


def align_field_to_target(source_field: xr.DataArray, target_field: xr.DataArray) -> xr.DataArray:
    return source_field.interp(
        latitude=target_field.latitude,
        longitude=target_field.longitude,
        method="nearest",
    )


def ensure_etopo1_file():
    if not ETOPO1_GZ_PATH.exists():
        print("Downloading NOAA ETOPO1 source file:", ETOPO1_URL)
        urllib.request.urlretrieve(ETOPO1_URL, ETOPO1_GZ_PATH)
        maybe_copy_to_drive(ETOPO1_GZ_PATH)
    if not ETOPO1_GRD_PATH.exists():
        print("Decompressing NOAA ETOPO1 source file:", ETOPO1_GRD_PATH.name)
        with gzip.open(ETOPO1_GZ_PATH, "rb") as src, open(ETOPO1_GRD_PATH, "wb") as dst:
            shutil.copyfileobj(src, dst)
        maybe_copy_to_drive(ETOPO1_GRD_PATH)


def infer_surface_elevation_field_from_etopo1():
    ensure_etopo1_file()
    ds = xr.open_dataset(ETOPO1_GRD_PATH)
    coord_renames = {}
    if "x" in ds.coords:
        coord_renames["x"] = "longitude"
    elif "lon" in ds.coords:
        coord_renames["lon"] = "longitude"
    if "y" in ds.coords:
        coord_renames["y"] = "latitude"
    elif "lat" in ds.coords:
        coord_renames["lat"] = "latitude"
    if coord_renames:
        ds = ds.rename(coord_renames)

    if "z" in ds.data_vars:
        elevation = ds["z"]
    else:
        two_d_vars = [name for name, data in ds.data_vars.items() if data.ndim >= 2]
        if not two_d_vars:
            raise RuntimeError(f"Could not find a 2-D elevation variable in {ETOPO1_GRD_PATH.name}; found {list(ds.data_vars)}")
        elevation = ds[two_d_vars[0]]

    elevation = strip_nonspatial_coords(elevation).astype(float).rename("surface_elevation_m")
    if "longitude" not in elevation.coords or "latitude" not in elevation.coords:
        raise RuntimeError(f"ETOPO1 subset did not expose longitude/latitude coordinates; found {list(elevation.coords)}")

    if float(elevation.longitude.min().values) < 0.0:
        elevation = elevation.assign_coords(longitude=((elevation.longitude + 360.0) % 360.0)).sortby("longitude")

    elevation_subset = subset_to_bounds(
        elevation,
        OBJECTIVE_SUBTYPE_DOMAIN.lon_min,
        OBJECTIVE_SUBTYPE_DOMAIN.lon_max,
        OBJECTIVE_SUBTYPE_DOMAIN.lat_min,
        OBJECTIVE_SUBTYPE_DOMAIN.lat_max,
    ).load()
    front_box_subset = subset_to_box(elevation_subset, HOKKAIDO_FRONT_BOX)

    diagnostics_df = pd.DataFrame([
        {
            "source": "NOAA ETOPO1 Ice Surface",
            "source_url": ETOPO1_URL,
            "cached_grd_path": str(ETOPO1_GRD_PATH),
            "source_variable": getattr(elevation, "name", "z"),
            "source_units": str(elevation.attrs.get("units", "m")),
            "conversion": "native meters",
            "domain_finite_fraction": round(float(np.isfinite(elevation_subset.values).mean()), 3),
            "front_box_finite_fraction": round(float(np.isfinite(front_box_subset.values).mean()), 3),
            "domain_min_elevation_m": round(float(np.nanmin(elevation_subset.values)), 2),
            "domain_max_elevation_m": round(float(np.nanmax(elevation_subset.values)), 2),
        }
    ])

    elevation_subset.attrs["source"] = "NOAA ETOPO1 Ice Surface"
    elevation_subset.attrs["source_url"] = ETOPO1_URL
    elevation_subset.attrs["source_variable"] = getattr(elevation, "name", "z")
    elevation_subset.attrs["source_units"] = str(elevation.attrs.get("units", "m"))
    elevation_subset.attrs["conversion"] = "native meters"
    elevation_subset.attrs["units"] = "m"
    return elevation_subset, diagnostics_df


In [ ]:
for path_to_restore in [CLUSTERED_K3_PATH]:
    if not path_to_restore.exists():
        restore_from_drive_cache(CLUSTERED_K3_PATH)

clustered_k3_df = pd.read_csv(CLUSTERED_K3_PATH)
clustered_k3_df = add_japan_local_time_columns(clustered_k3_df)
if PRIMARY_CLUSTER_COLUMN not in clustered_k3_df.columns:
    cluster_cols = [column for column in clustered_k3_df.columns if column.startswith("cluster_")]
    raise RuntimeError(f"Expected {PRIMARY_CLUSTER_COLUMN} in clustered_events_k3.csv, found {cluster_cols}")

current_cluster_labels = clustered_k3_df[PRIMARY_CLUSTER_COLUMN].astype(int)
current_cluster_counts_df = cluster_count_table(current_cluster_labels)
current_cluster_label_lookup = current_cluster_counts_df.set_index("cluster_id")["cluster_label"].to_dict()

print("Loaded current clustered k=3 rerun table from Notebook 08 outputs")
display(current_cluster_counts_df)
print("\nThis notebook keeps the current 4-feature solution as the baseline and replaces only the T850 frontality metric with elevation-masked versions.")


In [ ]:
for path_to_restore in [SURFACE_ELEVATION_CACHE_PATH, SURFACE_ELEVATION_DIAGNOSTICS_PATH, THRESHOLD_SUMMARY_PATH]:
    if not path_to_restore.exists():
        restore_from_drive_cache(path_to_restore)

surface_elevation_ds = None
surface_elevation_diagnostics_df = None
if SURFACE_ELEVATION_CACHE_PATH.exists() and not FORCE_REBUILD_SURFACE_ELEVATION:
    cached_surface_elevation_ds = xr.open_dataset(SURFACE_ELEVATION_CACHE_PATH).load()
    cached_surface_elevation_field = cached_surface_elevation_ds["surface_elevation_m"]
    cached_source = str(cached_surface_elevation_field.attrs.get("source", ""))
    cached_domain_finite_fraction = float(np.isfinite(cached_surface_elevation_field.values).mean())
    cached_front_box = subset_to_box(cached_surface_elevation_field, HOKKAIDO_FRONT_BOX)
    cached_front_box_finite_fraction = float(np.isfinite(cached_front_box.values).mean()) if cached_front_box.size else 0.0
    if "ETOPO1" in cached_source and cached_domain_finite_fraction > 0.0 and cached_front_box_finite_fraction > 0.0:
        surface_elevation_ds = cached_surface_elevation_ds
        print("Loaded cached NOAA ETOPO1 surface-elevation dataset")
    else:
        print(
            "Cached surface-elevation dataset is missing NOAA ETOPO1 metadata or finite coverage in the Hokkaido front box, "
            "so it will be rebuilt from NOAA ETOPO1."
        )
if SURFACE_ELEVATION_DIAGNOSTICS_PATH.exists() and not FORCE_REBUILD_SURFACE_ELEVATION:
    surface_elevation_diagnostics_df = pd.read_csv(SURFACE_ELEVATION_DIAGNOSTICS_PATH)

if surface_elevation_ds is None or FORCE_REBUILD_SURFACE_ELEVATION:
    surface_elevation_field, surface_elevation_diagnostics_df = infer_surface_elevation_field_from_etopo1()
    surface_elevation_ds = xr.Dataset({"surface_elevation_m": surface_elevation_field})
    surface_elevation_ds.to_netcdf(SURFACE_ELEVATION_CACHE_PATH)
    maybe_copy_to_drive(SURFACE_ELEVATION_CACHE_PATH)
    if surface_elevation_diagnostics_df is not None:
        surface_elevation_diagnostics_df.to_csv(SURFACE_ELEVATION_DIAGNOSTICS_PATH, index=False)
        maybe_copy_to_drive(SURFACE_ELEVATION_DIAGNOSTICS_PATH)

surface_elevation_field = surface_elevation_ds["surface_elevation_m"]
threshold_rows = []
for threshold_m in ELEVATION_THRESHOLDS_M:
    keep_mask = surface_elevation_field <= float(threshold_m)
    front_box_keep_mask = subset_to_box(keep_mask, HOKKAIDO_FRONT_BOX)
    threshold_rows.append(
        {
            "threshold_m": int(threshold_m),
            "source_variable": surface_elevation_field.attrs.get("source_variable", ""),
            "source_units": surface_elevation_field.attrs.get("source_units", ""),
            "conversion": surface_elevation_field.attrs.get("conversion", ""),
            "domain_min_elevation_m": round(float(np.nanmin(surface_elevation_field.values)), 2),
            "domain_max_elevation_m": round(float(np.nanmax(surface_elevation_field.values)), 2),
            "fraction_of_full_domain_kept": round(float(keep_mask.mean().values), 3),
            "fraction_of_hokkaido_front_box_kept": round(float(front_box_keep_mask.mean().values), 3),
        }
    )
threshold_summary_df = pd.DataFrame(threshold_rows)
threshold_summary_df.to_csv(THRESHOLD_SUMMARY_PATH, index=False)
maybe_copy_to_drive(THRESHOLD_SUMMARY_PATH)

print("Surface-elevation candidate diagnostics")
if surface_elevation_diagnostics_df is not None:
    display(surface_elevation_diagnostics_df)
print("\nElevation-mask threshold summary")
display(threshold_summary_df)


In [ ]:
if 'threshold_summary_df' not in globals() or 'surface_elevation_field' not in globals():
    raise RuntimeError(
        "Run the surface-elevation summary cell first so the elevation field and threshold table are defined before the sanity check."
    )

if 'era5_runtime_ds' not in globals():
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

sample_event_row = clustered_k3_df.iloc[0].copy()
sample_temp_snapshot_850 = load_offset_snapshot(
    era5_runtime_ds,
    sample_event_row["event_peak"],
    offset_hours=0,
    variables=["temperature"],
    domain=OBJECTIVE_SUBTYPE_DOMAIN,
    level=850,
)
sample_temp_grad = compute_temperature_gradient_magnitude(sample_temp_snapshot_850)
sample_temp_grad_display = (sample_temp_grad * float(sample_temp_grad.attrs.get("display_scale_factor", 1.0))).rename("temperature_gradient_display")
aligned_surface_elevation = align_field_to_target(surface_elevation_field, sample_temp_grad_display)

sanity_rows = []
for threshold_m in ELEVATION_THRESHOLDS_M:
    keep_mask = aligned_surface_elevation <= float(threshold_m)
    masked_temp_grad = sample_temp_grad_display.where(keep_mask)
    sample_front_box_masked = subset_to_box(masked_temp_grad, HOKKAIDO_FRONT_BOX)
    sample_front_box_unmasked = subset_to_box(sample_temp_grad_display, HOKKAIDO_FRONT_BOX)
    sanity_rows.append(
        {
            "threshold_m": int(threshold_m),
            "sample_event_peak": str(pd.Timestamp(sample_event_row["event_peak"])),
            "sample_current_cluster": int(sample_event_row[PRIMARY_CLUSTER_COLUMN]),
            "sample_offset_hours": 0,
            "keep_fraction_full_domain": round(float(keep_mask.mean().values), 3),
            "keep_fraction_hokkaido_front_box": round(float(sample_front_box_masked.notnull().mean().values), 3),
            "sample_front_box_unmasked_max": round(float(sample_front_box_unmasked.max(skipna=True).values), 3),
            "sample_front_box_masked_max": round(float(sample_front_box_masked.max(skipna=True).values), 3),
            "sample_front_box_finite_cell_count_after_mask": int(sample_front_box_masked.notnull().sum().values),
        }
    )

sanity_df = pd.DataFrame(sanity_rows)
print("One-event elevation-mask T850 sanity check before the full event loop")
display(sanity_df)
if int(sanity_df["sample_front_box_finite_cell_count_after_mask"].min()) <= 0:
    raise RuntimeError(
        "At least one tested elevation threshold removes all valid cells from the Hokkaido front box for the sanity-check event. "
        "Do not run the full masked-feature loop until the thresholds or elevation field are corrected."
    )


In [ ]:
for path_to_restore in [MASKED_EVENT_FEATURES_PATH, MASKED_EVENT_FEATURES_PARTIAL_PATH]:
    if not path_to_restore.exists():
        restore_from_drive_cache(path_to_restore)

if 'era5_runtime_ds' not in globals():
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

existing_masked_df = None
if MASKED_EVENT_FEATURES_PATH.exists() and not FORCE_REBUILD_MASKED_T850:
    existing_masked_df = pd.read_csv(MASKED_EVENT_FEATURES_PATH)
    print("Loaded cached final elevation-masked T850 event feature table")
elif MASKED_EVENT_FEATURES_PARTIAL_PATH.exists() and not FORCE_REBUILD_MASKED_T850:
    existing_masked_df = pd.read_csv(MASKED_EVENT_FEATURES_PARTIAL_PATH)
    print("Loaded cached partial elevation-masked T850 event feature table")

masked_feature_columns = [MASKED_FEATURE_COLUMN_BY_THRESHOLD[int(threshold_m)] for threshold_m in ELEVATION_THRESHOLDS_M]
if existing_masked_df is not None and not existing_masked_df.empty:
    existing_masked_df["event_row_index"] = pd.to_numeric(existing_masked_df["event_row_index"], errors="coerce").astype("Int64")
    for column_name in masked_feature_columns:
        existing_masked_df[column_name] = pd.to_numeric(existing_masked_df[column_name], errors="coerce")
    finite_cached = int(existing_masked_df[masked_feature_columns].notna().any(axis=1).sum())
    print(f"Cached elevation-masked T850 rows with at least one finite feature value: {finite_cached}/{len(existing_masked_df)}")
    if finite_cached == 0:
        print("Cached elevation-masked T850 table has zero finite values, so it will be ignored and rebuilt.")
        existing_masked_df = None

processed_records = []
processed_event_indices = set()
if existing_masked_df is not None and not existing_masked_df.empty:
    processed_records = existing_masked_df.to_dict(orient="records")
    processed_event_indices = {int(value) for value in existing_masked_df["event_row_index"].dropna().tolist()}

aligned_surface_elevation_850 = None

def compute_masked_t850_features_for_event(row: pd.Series) -> dict[str, float]:
    global aligned_surface_elevation_850
    per_threshold_values = {int(threshold_m): [] for threshold_m in ELEVATION_THRESHOLDS_M}
    for offset_hours in OFFSET_HOURS:
        temp_snapshot_850 = load_offset_snapshot(
            era5_runtime_ds,
            row["event_peak"],
            offset_hours=offset_hours,
            variables=["temperature"],
            domain=OBJECTIVE_SUBTYPE_DOMAIN,
            level=850,
        )
        temp_grad = compute_temperature_gradient_magnitude(temp_snapshot_850)
        temp_grad_display = (temp_grad * float(temp_grad.attrs.get("display_scale_factor", 1.0))).rename("temperature_gradient_display")
        if aligned_surface_elevation_850 is None or aligned_surface_elevation_850.shape != temp_grad_display.shape:
            aligned_surface_elevation_850 = align_field_to_target(surface_elevation_field, temp_grad_display)
        for threshold_m in ELEVATION_THRESHOLDS_M:
            masked_temp_grad = temp_grad_display.where(aligned_surface_elevation_850 <= float(threshold_m))
            per_threshold_values[int(threshold_m)].append(_box_max(masked_temp_grad, HOKKAIDO_FRONT_BOX))
    return {
        MASKED_FEATURE_COLUMN_BY_THRESHOLD[int(threshold_m)]: safe_nanmax(values)
        for threshold_m, values in per_threshold_values.items()
    }

remaining_rows = clustered_k3_df.loc[~clustered_k3_df.index.isin(processed_event_indices)].copy()
print(f"Elevation-masked T850 feature progress before this run: {len(processed_event_indices)}/{len(clustered_k3_df)} events")

for _, (row_index, row) in enumerate(remaining_rows.iterrows(), start=1):
    threshold_feature_values = compute_masked_t850_features_for_event(row)
    record = {
        "event_row_index": int(row_index),
        "event_peak": pd.Timestamp(row["event_peak"]),
        "current_cluster_id": int(row[PRIMARY_CLUSTER_COLUMN]),
        "current_cluster_label": current_cluster_label_lookup[int(row[PRIMARY_CLUSTER_COLUMN])],
        "unmasked_t850_feature": float(row[DROPPED_FEATURE]),
    }
    for threshold_m in ELEVATION_THRESHOLDS_M:
        column_name = MASKED_FEATURE_COLUMN_BY_THRESHOLD[int(threshold_m)]
        record[column_name] = threshold_feature_values[column_name]
        record[f"masked_{int(threshold_m)}m_minus_unmasked_t850"] = (
            threshold_feature_values[column_name] - float(row[DROPPED_FEATURE])
            if np.isfinite(threshold_feature_values[column_name])
            else np.nan
        )
    processed_records.append(record)

    event_number = len(processed_records)
    if event_number % CHECKPOINT_EVERY_EVENTS == 0 or event_number == len(clustered_k3_df):
        checkpoint_df = pd.DataFrame(processed_records).sort_values("event_row_index").reset_index(drop=True)
        checkpoint_df.to_csv(MASKED_EVENT_FEATURES_PARTIAL_PATH, index=False)
        maybe_copy_to_drive(MASKED_EVENT_FEATURES_PARTIAL_PATH)
        print(f"Checkpointed elevation-masked T850 event feature table at {event_number}/{len(clustered_k3_df)} events")

masked_event_feature_df = pd.DataFrame(processed_records).sort_values("event_row_index").reset_index(drop=True)
masked_event_feature_df["event_row_index"] = pd.to_numeric(masked_event_feature_df["event_row_index"], errors="coerce").astype(int)
for column_name in masked_feature_columns:
    masked_event_feature_df[column_name] = pd.to_numeric(masked_event_feature_df[column_name], errors="coerce")
finite_summary_rows = []
for threshold_m in ELEVATION_THRESHOLDS_M:
    column_name = MASKED_FEATURE_COLUMN_BY_THRESHOLD[int(threshold_m)]
    finite_count = int(masked_event_feature_df[column_name].notna().sum())
    finite_summary_rows.append({"threshold_m": int(threshold_m), "finite_event_count": finite_count})
finite_summary_df = pd.DataFrame(finite_summary_rows)
print("Finite elevation-masked T850 feature values available after this run")
display(finite_summary_df)
if int(finite_summary_df["finite_event_count"].min()) == 0:
    raise RuntimeError(
        "At least one tested elevation threshold produced zero finite masked T850 feature values. "
        "Revisit the elevation field or threshold list before continuing."
    )
masked_event_feature_df.to_csv(MASKED_EVENT_FEATURES_PATH, index=False)
maybe_copy_to_drive(MASKED_EVENT_FEATURES_PATH)
print("Saved final elevation-masked T850 event feature table")

feature_comparison_rows = []
for threshold_m in ELEVATION_THRESHOLDS_M:
    column_name = MASKED_FEATURE_COLUMN_BY_THRESHOLD[int(threshold_m)]
    diff_column = f"masked_{int(threshold_m)}m_minus_unmasked_t850"
    threshold_summary = (
        masked_event_feature_df.groupby("current_cluster_id")
        .agg(
            cluster_label=("current_cluster_label", "first"),
            n_events=("event_row_index", "count"),
            unmasked_t850_median=("unmasked_t850_feature", "median"),
            masked_t850_median=(column_name, "median"),
            median_masked_minus_unmasked=(diff_column, "median"),
            mean_abs_masked_minus_unmasked=(diff_column, lambda s: float(np.nanmean(np.abs(s)))),
        )
        .reset_index()
        .rename(columns={"current_cluster_id": "cluster_id"})
    )
    threshold_summary["threshold_m"] = int(threshold_m)
    feature_comparison_rows.append(threshold_summary)
feature_comparison_df = pd.concat(feature_comparison_rows, ignore_index=True)
for column_name in ["unmasked_t850_median", "masked_t850_median", "median_masked_minus_unmasked", "mean_abs_masked_minus_unmasked"]:
    feature_comparison_df[column_name] = feature_comparison_df[column_name].round(3)
feature_comparison_df.to_csv(FEATURE_COMPARISON_PATH, index=False)
maybe_copy_to_drive(FEATURE_COMPARISON_PATH)

print("Per-current-cluster comparison of unmasked versus elevation-masked T850 feature")
display(feature_comparison_df)


In [ ]:
baseline_standardized_df, baseline_scores_df, baseline_variance_df, baseline_loadings_df, baseline_driver_df, baseline_feature_means, baseline_feature_stds = compute_pca_diagnostics(
    clustered_k3_df,
    FULL_FEATURE_COLUMNS,
)

switching_rows = []
crosstab_rows = []
count_rows = []
median_rows = []
quality_rows = []
variance_rows = [baseline_variance_df.assign(solution="current_4_feature_k3")]
loadings_rows = [baseline_loadings_df.assign(solution="current_4_feature_k3")]
driver_rows = [baseline_driver_df.assign(solution="current_4_feature_k3")]
solution_rows = [
    {
        "solution": "current_4_feature_k3",
        "threshold_m": np.nan,
        "n_features": len(FULL_FEATURE_COLUMNS),
        "features": ", ".join(FULL_FEATURE_COLUMNS),
        "silhouette": float(compute_mean_silhouette_score(baseline_standardized_df, current_cluster_labels)),
        "cluster_counts": ", ".join([f"{int(row.cluster_id)}:{int(row.n_events)}" for row in current_cluster_counts_df.itertuples(index=False)]),
    }
]
clustered_events_export_df = clustered_k3_df.copy()

for threshold_m in ELEVATION_THRESHOLDS_M:
    threshold_key = int(threshold_m)
    feature_column = MASKED_FEATURE_COLUMN_BY_THRESHOLD[threshold_key]
    cluster_column_raw = MASKED_CLUSTER_COLUMN_RAW_BY_THRESHOLD[threshold_key]
    cluster_column = MASKED_CLUSTER_COLUMN_BY_THRESHOLD[threshold_key]
    feature_columns = MASKED_FEATURE_COLUMNS_BY_THRESHOLD[threshold_key]

    threshold_feature_df = clustered_k3_df.copy()
    masked_event_feature_df["event_row_index"] = pd.to_numeric(masked_event_feature_df["event_row_index"], errors="coerce").astype(int)
    threshold_lookup = masked_event_feature_df.set_index("event_row_index")[feature_column]
    threshold_feature_df[feature_column] = pd.to_numeric(threshold_lookup.reindex(threshold_feature_df.index), errors="coerce")
    finite_rows = int(threshold_feature_df[feature_column].notna().sum())
    print(f"Rows with finite elevation-masked T850 values entering PCA/clustering for {threshold_key} m: {finite_rows}/{len(threshold_feature_df)}")
    if finite_rows == 0:
        raise RuntimeError(f"No finite elevation-masked T850 rows are available for the {threshold_key} m threshold.")

    standardized_df, scores_df, variance_df, loadings_df, driver_df, _, _ = compute_pca_diagnostics(
        threshold_feature_df,
        feature_columns,
    )
    quality_df = evaluate_hierarchical_cluster_solutions(
        standardized_df,
        cluster_counts=CLUSTER_COUNT_OPTIONS,
        method="ward",
    ).assign(threshold_m=threshold_key)
    cluster_labels_raw = assign_hierarchical_clusters(
        standardized_df,
        n_clusters=PRIMARY_CLUSTER_K,
        method="ward",
    ).rename(cluster_column_raw)
    label_mapping, best_match_count = best_cluster_label_mapping(current_cluster_labels.loc[cluster_labels_raw.index], cluster_labels_raw)
    cluster_labels = cluster_labels_raw.map(label_mapping).rename(cluster_column)

    clustered_events_export_df[feature_column] = threshold_feature_df[feature_column]
    clustered_events_export_df[cluster_column_raw] = cluster_labels_raw.reindex(clustered_events_export_df.index)
    clustered_events_export_df[cluster_column] = cluster_labels.reindex(clustered_events_export_df.index)

    switch_mask = cluster_labels != current_cluster_labels.loc[cluster_labels.index]
    switching_rows.append(
        {
            "threshold_m": threshold_key,
            "cluster_id": "all",
            "cluster_label": "All events",
            "n_events": int(len(cluster_labels)),
            "n_switched": int(switch_mask.sum()),
            "percent_switched": float(100.0 * switch_mask.mean()),
            "best_label_mapping": str(label_mapping),
            "best_label_match_count": int(best_match_count),
        }
    )
    for cluster_id in sorted(current_cluster_labels.dropna().astype(int).unique()):
        cluster_mask = current_cluster_labels.loc[cluster_labels.index] == int(cluster_id)
        switching_rows.append(
            {
                "threshold_m": threshold_key,
                "cluster_id": int(cluster_id),
                "cluster_label": current_cluster_label_lookup[int(cluster_id)],
                "n_events": int(cluster_mask.sum()),
                "n_switched": int(switch_mask.loc[cluster_mask].sum()),
                "percent_switched": float(100.0 * switch_mask.loc[cluster_mask].mean()),
                "best_label_mapping": str(label_mapping),
                "best_label_match_count": int(best_match_count),
            }
        )

    crosstab_df = pd.crosstab(
        current_cluster_labels.rename("current_4_feature_cluster"),
        cluster_labels.rename(f"masked_t850_{threshold_key}m_cluster"),
    ).reset_index()
    crosstab_df["threshold_m"] = threshold_key
    crosstab_rows.append(crosstab_df)

    counts_df = cluster_count_table(cluster_labels)
    counts_df["threshold_m"] = threshold_key
    count_rows.append(counts_df)

    medians_df = (
        threshold_feature_df.loc[cluster_labels.index]
        .groupby(cluster_labels)[[
            "coastal_to_jpcz_mean_divergence_ratio",
            "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
            DROPPED_FEATURE,
            feature_column,
            "sea_of_japan_mean_vorticity_peak_925",
        ]]
        .median()
        .round(3)
        .reset_index()
        .rename(columns={cluster_column: "cluster_id"})
    )
    medians_df["threshold_m"] = threshold_key
    median_rows.append(medians_df)

    solution_rows.append(
        {
            "solution": f"masked_t850_{threshold_key}m_k3",
            "threshold_m": threshold_key,
            "n_features": len(feature_columns),
            "features": ", ".join(feature_columns),
            "silhouette": float(compute_mean_silhouette_score(standardized_df, cluster_labels)),
            "cluster_counts": ", ".join([f"{int(row.cluster_id)}:{int(row.n_events)}" for row in counts_df.itertuples(index=False)]),
        }
    )

    variance_rows.append(variance_df.assign(solution=f"masked_t850_{threshold_key}m_k3", threshold_m=threshold_key))
    loadings_rows.append(loadings_df.assign(solution=f"masked_t850_{threshold_key}m_k3", threshold_m=threshold_key))
    driver_rows.append(driver_df.assign(solution=f"masked_t850_{threshold_key}m_k3", threshold_m=threshold_key))
    quality_rows.append(quality_df)

switching_df = pd.DataFrame(switching_rows)
switching_df["percent_switched"] = switching_df["percent_switched"].round(2)
counts_df_long = pd.concat(count_rows, ignore_index=True)
medians_df_long = pd.concat(median_rows, ignore_index=True)
quality_df_long = pd.concat(quality_rows, ignore_index=True)
solution_summary_df = pd.DataFrame(solution_rows)
solution_summary_df["silhouette"] = solution_summary_df["silhouette"].round(5)
combined_variance_df = pd.concat(variance_rows, ignore_index=True)
combined_loadings_df = pd.concat(loadings_rows, ignore_index=True)
combined_driver_df = pd.concat(driver_rows, ignore_index=True)
crosstab_df_long = pd.concat(crosstab_rows, ignore_index=True)

for output_path, table_df in [
    (SWITCHING_PATH, switching_df),
    (CROSSTAB_PATH, crosstab_df_long),
    (COUNTS_PATH, counts_df_long),
    (MEDIANS_PATH, medians_df_long),
    (SOLUTION_SUMMARY_PATH, solution_summary_df),
    (QUALITY_SCAN_PATH, quality_df_long),
    (VARIANCE_PATH, combined_variance_df),
    (LOADINGS_PATH, combined_loadings_df),
    (DRIVERS_PATH, combined_driver_df),
    (CLUSTERED_EVENTS_PATH, clustered_events_export_df),
]:
    table_df.to_csv(output_path, index=False)
    maybe_copy_to_drive(output_path)

print("Current 4-feature versus elevation-masked T850 k=3 comparison")
display(solution_summary_df)
print("\nElevation-masked 4-feature quality scan across k = 2, 3, 4")
display(quality_df_long)
print("\nPercent of events that switch clusters after replacing T850 with the elevation-masked version and best-matching the labels")
display(switching_df)
print("\nCurrent 4-feature versus elevation-masked cluster cross-tabs")
display(crosstab_df_long)
print("\nElevation-masked cluster medians, including both the original and masked T850 feature columns for comparison")
display(medians_df_long)
print("\nPCA variance comparison")
display(combined_variance_df)
print("\nTop PCA loading drivers")
display(combined_driver_df)


In [ ]:
cluster_colors = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)
plot_thresholds = [int(threshold_m) for threshold_m in ELEVATION_THRESHOLDS_M]
for ax, threshold_key in zip(axes, plot_thresholds):
    feature_column = MASKED_FEATURE_COLUMN_BY_THRESHOLD[threshold_key]
    feature_columns = MASKED_FEATURE_COLUMNS_BY_THRESHOLD[threshold_key]
    threshold_feature_df = clustered_k3_df.copy()
    threshold_lookup = masked_event_feature_df.set_index("event_row_index")[feature_column]
    threshold_feature_df[feature_column] = pd.to_numeric(threshold_lookup.reindex(threshold_feature_df.index), errors="coerce")
    standardized_df, scores_df, variance_df, _, _, _, _ = compute_pca_diagnostics(threshold_feature_df, feature_columns)
    cluster_labels = clustered_events_export_df[MASKED_CLUSTER_COLUMN_BY_THRESHOLD[threshold_key]].dropna().astype(int)
    labels_for_scores = cluster_labels.loc[scores_df.index]
    for cluster_id in sorted(labels_for_scores.unique()):
        mask = labels_for_scores == cluster_id
        ax.scatter(
            scores_df.loc[mask, "PC1"],
            scores_df.loc[mask, "PC2"],
            s=24,
            alpha=0.8,
            color=cluster_colors.get(int(cluster_id), "gray"),
            label=f"Cluster {int(cluster_id)}",
        )
    pc1_pct = float(variance_df.loc[variance_df["principal_component"] == "PC1", "explained_variance_percent"].iloc[0])
    pc2_pct = float(variance_df.loc[variance_df["principal_component"] == "PC2", "explained_variance_percent"].iloc[0])
    ax.set_xlabel(f"PC1 ({pc1_pct:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({pc2_pct:.1f}% variance)")
    ax.set_title(f"Elevation mask <= {threshold_key} m")
    ax.grid(alpha=0.25)
axes[0].legend(loc="best")
fig.suptitle("PCA comparison across elevation-masked T850 thresholds", fontsize=13)
if SAVE_PLOTS:
    fig.savefig(PCA_SCATTER_PATH, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(PCA_SCATTER_PATH, verbose=False)
plt.show()

summary_fig, summary_axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
overall_switching = switching_df.loc[switching_df["cluster_id"] == "all"].copy().sort_values("threshold_m")
summary_axes[0].plot(overall_switching["threshold_m"], overall_switching["percent_switched"], marker="o")
summary_axes[0].set_xlabel("Elevation mask threshold [m]")
summary_axes[0].set_ylabel("Percent switched [%]")
summary_axes[0].set_title("Cluster-switch sensitivity")
summary_axes[0].grid(alpha=0.25)

masked_solution_summary = solution_summary_df.dropna(subset=["threshold_m"]).sort_values("threshold_m")
summary_axes[1].plot(masked_solution_summary["threshold_m"], masked_solution_summary["silhouette"], marker="o", label="Masked k=3 silhouette")
current_silhouette = float(solution_summary_df.loc[solution_summary_df["solution"] == "current_4_feature_k3", "silhouette"].iloc[0])
summary_axes[1].axhline(current_silhouette, color="gray", linestyle="--", label="Current k=3 silhouette")
summary_axes[1].set_xlabel("Elevation mask threshold [m]")
summary_axes[1].set_ylabel("Mean silhouette score")
summary_axes[1].set_title("k=3 silhouette sensitivity")
summary_axes[1].grid(alpha=0.25)
summary_axes[1].legend(loc="best")
if SAVE_PLOTS:
    summary_fig.savefig(SENSITIVITY_SUMMARY_PLOT_PATH, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(SENSITIVITY_SUMMARY_PLOT_PATH, verbose=False)
plt.show()

boxplot_fig, boxplot_axes = plt.subplots(1, len(plot_thresholds), figsize=(18, 5.5), constrained_layout=True)
if len(plot_thresholds) == 1:
    boxplot_axes = [boxplot_axes]
for ax, threshold_key in zip(boxplot_axes, plot_thresholds):
    feature_column = MASKED_FEATURE_COLUMN_BY_THRESHOLD[threshold_key]
    plot_feature_df = masked_event_feature_df.copy()
    plot_feature_df["current_cluster_id"] = plot_feature_df["current_cluster_id"].astype(int)
    positions = []
    labels = []
    boxplot_data = []
    colors = []
    for cluster_id in sorted(plot_feature_df["current_cluster_id"].unique()):
        cluster_subset = plot_feature_df.loc[plot_feature_df["current_cluster_id"] == cluster_id]
        positions.extend([cluster_id - 0.15, cluster_id + 0.15])
        labels.extend([f"C{cluster_id}\nunmasked", f"C{cluster_id}\nmasked"])
        boxplot_data.extend([
            cluster_subset["unmasked_t850_feature"].dropna().values,
            cluster_subset[feature_column].dropna().values,
        ])
        colors.extend(["#9ecae1", "#fdae6b"])
    box = ax.boxplot(boxplot_data, positions=positions, widths=0.22, patch_artist=True, showfliers=False)
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor(color)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_title(f"Mask <= {threshold_key} m")
    ax.grid(axis="y", alpha=0.25)
boxplot_axes[0].set_ylabel("Front-box maximum |grad T850| [K (100 km)^-1]")
if SAVE_PLOTS:
    boxplot_fig.savefig(T850_COMPARE_PLOT_PATH, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(T850_COMPARE_PLOT_PATH, verbose=False)
plt.show()

print("Saved elevation-mask T850 sensitivity plots")
display(pd.DataFrame({
    "plot": ["PCA scatters by threshold", "Threshold sensitivity summary", "Current-cluster T850 unmasked versus masked boxplots"],
    "path": [str(PCA_SCATTER_PATH), str(SENSITIVITY_SUMMARY_PLOT_PATH), str(T850_COMPARE_PLOT_PATH)],
}))


### How to read the result

- If one or more elevation thresholds reduce the terrain-linked `T850` influence while preserving a similar `k = 3` structure, then that masked frontality metric is a stronger candidate for the main workflow than the unmasked version.
- If the masked thresholds produce large switching or collapse the refined split, then the current `k = 3` story depends strongly on the terrain-influenced part of the frontality metric.
- Compare `800 / 1000 / 1200 m` together rather than treating a single threshold as final by default.
